# Chapter 25 Lab: AI-Assisted Time-Series Modeling

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-25-ai-assisted-time-series-modeling-lab.ipynb)

This independent-study lab shows how to use an AI assistant as a modeling partner while keeping mathematical and empirical control of the forecasting workflow. No external AI API is required. Instead, you will practice the parts of AI-assisted modeling that students can verify directly:

1. define the forecasting task and information set;
2. build leakage-safe features;
3. compare against simple baselines;
4. test AI-suggested workflows for leakage and invalid validation;
5. diagnose residuals and horizon-specific errors;
6. write prompts that ask for verification steps instead of unsupported recommendations;
7. produce a short modeling log that separates assumptions, evidence, limitations, and decisions.

The guiding principle is:

$$
\text{AI proposes; the analyst verifies.}
$$

Because some LaTeX commands do not display reliably in all Colab environments, this notebook avoids specialized calligraphic notation.


## 0. Mathematical and practical background

A forecasting model should only use information available at the forecast origin. If the origin is time $t$ and the horizon is $h$, the prediction target is $y_{t+h}$. A valid forecast may use past observations, calendar variables known in advance, and planned events known at time $t$. It should not use future observed target values.

Examples of valid features for predicting $y_{t+h}$:

- $y_t$, $y_{t-1}$, $y_{t-2}$;
- trailing rolling averages such as the mean of $y_{t-6}, \ldots, y_t$;
- target-date calendar variables such as day of week, month, or known holiday schedule.

Examples of invalid features:

- a centered rolling average around the target date;
- a scaler fitted using the full dataset before the train/test split;
- a covariate whose value would not be known at the forecast origin;
- a random train/test split used to evaluate a future-forecasting task.

In this lab, you will intentionally create both safe and unsafe workflows, then audit them.


## 1. Setup

We use standard packages available in Colab. The models are intentionally modest. The goal is methodology, not model complexity.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from scipy.stats import chi2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(7339)
pd.set_option("display.max_columns", 30)


## 2. Simulate a forecasting project

We create a daily demand series with trend, weekly seasonality, annual seasonality, planned promotions, a changepoint, and autocorrelated noise. This lets us practice realistic auditing without needing an external dataset.

The response variable is demand. The forecast horizon is seven days ahead.


In [ ]:
def simulate_daily_demand(n=730, seed=7339):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2022-01-01", periods=n, freq="D")
    t = np.arange(n)

    weekly = 8.0 * np.sin(2 * np.pi * t / 7) + 3.0 * np.cos(2 * np.pi * t / 7)
    yearly = 5.0 * np.sin(2 * np.pi * t / 365.25)
    trend = 80 + 0.035 * t + 0.045 * np.maximum(t - 430, 0)

    # A planned promotion schedule. This is known ahead of time.
    promo = (((t % 55) >= 8) & ((t % 55) <= 13)).astype(float)

    # A holiday-like calendar effect. Also known ahead of time.
    holiday = ((dates.month == 12) & (dates.day >= 20) & (dates.day <= 26)).astype(float)

    eps = rng.normal(0, 2.2, n)
    ar_noise = np.zeros(n)
    for i in range(1, n):
        ar_noise[i] = 0.55 * ar_noise[i - 1] + eps[i]

    y = trend + weekly + yearly + 8.0 * promo + 12.0 * holiday + ar_noise

    return pd.DataFrame({
        "ds": dates,
        "t": t,
        "y": y,
        "promo": promo,
        "holiday": holiday,
        "dayofweek": dates.dayofweek,
        "month": dates.month,
    })

raw = simulate_daily_demand()
raw.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(raw["ds"], raw["y"], linewidth=1.2)
ax.set_title("Simulated daily demand")
ax.set_xlabel("date")
ax.set_ylabel("demand")
plt.show()


## 3. Define the forecasting task and information set

We will forecast seven days ahead. For a row with forecast origin index `origin`, the target is `origin + horizon`.

Safe features may use:

- target history through the origin date;
- trailing rolling summaries ending at the origin date;
- target-date calendar information that is known in advance;
- planned promotions or holidays known in advance.

Unsafe features may use future target values, future residuals, or future variables not known at the origin.


In [ ]:
HORIZON = 7
MAX_LAG = 21
ROLLING_WINDOWS = (7, 14, 28)
SEASON = 7

print(f"Forecast horizon: {HORIZON} days")
print(f"Maximum lag:      {MAX_LAG} days")
print(f"Rolling windows:  {ROLLING_WINDOWS}")


## 4. Build leakage-safe and intentionally leaky feature tables

The function below returns one row per forecast origin. Each row contains features known at the origin and the future target. We also include an option to add an intentionally leaky centered rolling feature for auditing practice.


In [ ]:
def build_forecast_table(data, horizon=7, max_lag=21, rolling_windows=(7, 14, 28), include_leaky=False):
    y = data["y"].to_numpy()
    n = len(data)
    min_origin = max(max_lag, max(rolling_windows) - 1, 2 * 7)
    rows = []

    for origin in range(min_origin, n - horizon):
        target_index = origin + horizon
        origin_date = data.loc[origin, "ds"]
        target_date = data.loc[target_index, "ds"]

        row = {
            "origin": origin,
            "target_index": target_index,
            "origin_date": origin_date,
            "target_date": target_date,
            "y_target": y[target_index],
        }

        # Past target values available at the forecast origin.
        row["y_lag_0"] = y[origin]
        for lag in range(1, max_lag + 1):
            row[f"y_lag_{lag}"] = y[origin - lag]

        # Trailing summaries ending at the origin date.
        for w in rolling_windows:
            window = y[origin - w + 1:origin + 1]
            row[f"roll_mean_{w}"] = window.mean()
            row[f"roll_std_{w}"] = window.std(ddof=0)

        # Target-date calendar variables. These are known before the target date.
        dow = target_date.dayofweek
        moy = target_date.month
        row["target_dow_sin"] = np.sin(2 * np.pi * dow / 7)
        row["target_dow_cos"] = np.cos(2 * np.pi * dow / 7)
        row["target_month_sin"] = np.sin(2 * np.pi * moy / 12)
        row["target_month_cos"] = np.cos(2 * np.pi * moy / 12)
        row["target_is_weekend"] = float(dow >= 5)

        # Planned future indicators. In this simulated example, these are known in advance.
        row["target_promo"] = data.loc[target_index, "promo"]
        row["target_holiday"] = data.loc[target_index, "holiday"]
        row["origin_time"] = origin
        row["target_time"] = target_index

        if include_leaky:
            # This is invalid: it uses target-day and post-target demand values.
            lo = max(0, target_index - 3)
            hi = min(n, target_index + 4)
            row["LEAK_centered_target_mean_7"] = y[lo:hi].mean()

        rows.append(row)

    return pd.DataFrame(rows)

safe_table = build_forecast_table(raw, HORIZON, MAX_LAG, ROLLING_WINDOWS, include_leaky=False)
leaky_table = build_forecast_table(raw, HORIZON, MAX_LAG, ROLLING_WINDOWS, include_leaky=True)

safe_table.head()


## 5. Unit tests for target alignment

A small unit test can catch many AI-generated coding mistakes. The feature `y_lag_0` should equal the observed value at the origin, and the target should equal the value seven days later.


In [ ]:
def assert_alignment(table, raw_data, horizon):
    sample_rows = [0, len(table) // 2, len(table) - 1]
    for row_id in sample_rows:
        row = table.iloc[row_id]
        origin = int(row["origin"])
        target_index = int(row["target_index"])

        assert target_index == origin + horizon
        assert np.isclose(row["y_lag_0"], raw_data.loc[origin, "y"])
        assert np.isclose(row["y_target"], raw_data.loc[target_index, "y"])

        roll7 = raw_data.loc[origin - 6:origin, "y"].mean()
        assert np.isclose(row["roll_mean_7"], roll7)

    return "All alignment checks passed."

assert_alignment(safe_table, raw, HORIZON)


## 6. Chronological split and baseline forecasts

Before trusting any AI-suggested model, compare it with baselines. For a seven-day horizon with weekly seasonality, `y_lag_0` is also the seasonal-naive forecast from the same weekday one week earlier.


In [ ]:
metadata_cols = ["origin", "target_index", "origin_date", "target_date", "y_target"]
safe_features = [c for c in safe_table.columns if c not in metadata_cols]
leaky_features = [c for c in leaky_table.columns if c not in metadata_cols]

cut = int(0.75 * len(safe_table))
train_safe = safe_table.iloc[:cut].copy()
test_safe = safe_table.iloc[cut:].copy()

X_train = train_safe[safe_features]
y_train = train_safe["y_target"]
X_test = test_safe[safe_features]
y_test = test_safe["y_target"]

print("Training rows:", len(train_safe))
print("Testing rows: ", len(test_safe))
print("Test period:  ", test_safe["target_date"].min().date(), "to", test_safe["target_date"].max().date())


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mase(y_true, y_pred, scale):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / scale)

# In-sample seasonal naive errors on the training target provide a scale for MASE.
train_naive_errors = np.abs(train_safe["y_target"].to_numpy() - train_safe["y_lag_0"].to_numpy())
mase_scale = train_naive_errors.mean()

baseline_predictions = {
    "persistence / seasonal naive": test_safe["y_lag_0"].to_numpy(),
    "trailing mean 7": test_safe["roll_mean_7"].to_numpy(),
    "trailing mean 28": test_safe["roll_mean_28"].to_numpy(),
}

rows = []
for name, pred in baseline_predictions.items():
    rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": rmse(y_test, pred),
        "MASE": mase(y_test, pred, mase_scale),
    })

baseline_metrics = pd.DataFrame(rows).sort_values("RMSE")
baseline_metrics


## 7. Fit safe candidate models

Now we fit two common AI-suggested models: a regularized linear model and a gradient boosting model. The split remains chronological and all preprocessing is fitted on the training block only.


In [ ]:
safe_models = {
    "ridge safe pipeline": Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=5.0)),
    ]),
    "gradient boosting safe": GradientBoostingRegressor(
        n_estimators=160,
        learning_rate=0.04,
        max_depth=3,
        random_state=7339,
    ),
    "random forest safe": RandomForestRegressor(
        n_estimators=120,
        max_depth=8,
        min_samples_leaf=4,
        random_state=7339,
        n_jobs=-1,
    ),
}

predictions = {}
model_rows = []

for name, model in safe_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    model_rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": rmse(y_test, pred),
        "MASE": mase(y_test, pred, mase_scale),
    })

all_metrics = pd.concat([baseline_metrics, pd.DataFrame(model_rows)], ignore_index=True).sort_values("RMSE")
all_metrics


In [ ]:
best_model_name = all_metrics.iloc[0]["model"]
if best_model_name in predictions:
    best_pred = predictions[best_model_name]
else:
    best_pred = baseline_predictions[best_model_name]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test_safe["target_date"], y_test, label="observed", linewidth=1.5)
ax.plot(test_safe["target_date"], baseline_predictions["persistence / seasonal naive"], label="seasonal naive", alpha=0.8)
ax.plot(test_safe["target_date"], best_pred, label=f"best: {best_model_name}", alpha=0.9)
ax.set_title("Forecasts on the chronological test period")
ax.set_xlabel("target date")
ax.set_ylabel("demand")
ax.legend()
plt.show()


## 8. Random split versus chronological split

A random split is a common AI-generated mistake because it is standard in many ordinary machine-learning tasks. For forecasting, it answers the wrong question: it mixes the future into training.


In [ ]:
# Same features and same model, but a random split.
X_all = safe_table[safe_features]
y_all = safe_table["y_target"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_all, y_all, test_size=(1 - 0.75), random_state=7339, shuffle=True
)

random_split_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=5.0)),
])
random_split_model.fit(X_train_r, y_train_r)
random_pred = random_split_model.predict(X_test_r)

chrono_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=5.0)),
])
chrono_model.fit(X_train, y_train)
chrono_pred = chrono_model.predict(X_test)

split_compare = pd.DataFrame([
    {"evaluation design": "random split", "RMSE": rmse(y_test_r, random_pred), "MAE": mean_absolute_error(y_test_r, random_pred)},
    {"evaluation design": "chronological split", "RMSE": rmse(y_test, chrono_pred), "MAE": mean_absolute_error(y_test, chrono_pred)},
])
split_compare


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(split_compare["evaluation design"], split_compare["RMSE"])
ax.set_title("Random split and chronological split answer different questions")
ax.set_ylabel("RMSE")
plt.xticks(rotation=15, ha="right")
plt.show()


## 9. Intentional leakage demonstration

Next we add an invalid centered rolling average around the target date. This feature uses values that would not be available at the forecast origin. If an AI assistant writes a feature like this, it may produce excellent-looking metrics for the wrong reason.


In [ ]:
train_leaky = leaky_table.iloc[:cut].copy()
test_leaky = leaky_table.iloc[cut:].copy()

X_train_leaky = train_leaky[leaky_features]
y_train_leaky = train_leaky["y_target"]
X_test_leaky = test_leaky[leaky_features]
y_test_leaky = test_leaky["y_target"]

leaky_model = GradientBoostingRegressor(
    n_estimators=160,
    learning_rate=0.04,
    max_depth=3,
    random_state=7339,
)
leaky_model.fit(X_train_leaky, y_train_leaky)
leaky_pred = leaky_model.predict(X_test_leaky)

leakage_compare = pd.DataFrame([
    {"workflow": "safe gradient boosting", "RMSE": rmse(y_test, predictions["gradient boosting safe"]), "MAE": mean_absolute_error(y_test, predictions["gradient boosting safe"])},
    {"workflow": "leaky centered target feature", "RMSE": rmse(y_test_leaky, leaky_pred), "MAE": mean_absolute_error(y_test_leaky, leaky_pred)},
    {"workflow": "seasonal naive baseline", "RMSE": rmse(y_test, baseline_predictions["persistence / seasonal naive"]), "MAE": mean_absolute_error(y_test, baseline_predictions["persistence / seasonal naive"])},
]).sort_values("RMSE")

leakage_compare


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(leakage_compare["workflow"], leakage_compare["RMSE"])
ax.set_title("Leaky features can make evaluation look artificially strong")
ax.set_ylabel("RMSE")
plt.xticks(rotation=20, ha="right")
plt.show()


## 10. Preprocessing leakage check

A scaler should be fitted on the training block only. Fitting it on the full dataset uses future distributional information. In some simple simulations the metric difference is small, but the workflow is still invalid.


In [ ]:
# Correct pipeline: scaler fitted only during fit on the training set.
correct = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=20.0)),
])
correct.fit(X_train, y_train)
correct_pred = correct.predict(X_test)

# Incorrect workflow: scaler fitted on the entire dataset before the split.
full_scaler = StandardScaler().fit(X_all)
X_all_scaled = full_scaler.transform(X_all)
X_train_bad = X_all_scaled[:cut]
X_test_bad = X_all_scaled[cut:]

bad_model = Ridge(alpha=20.0)
bad_model.fit(X_train_bad, y_train)
bad_pred = bad_model.predict(X_test_bad)

pd.DataFrame([
    {"workflow": "correct train-only scaling", "RMSE": rmse(y_test, correct_pred)},
    {"workflow": "incorrect full-data scaling", "RMSE": rmse(y_test, bad_pred)},
])


## 11. Feature timing audit

A useful way to review AI-generated feature code is to record each feature's latest required source time relative to the forecast origin. Negative numbers and zero mean past or present information. Positive numbers mean future information. Future calendar variables can be allowed only when they are known in advance.


In [ ]:
def build_feature_provenance(features, horizon):
    records = []
    for feature in features:
        if feature.startswith("y_lag_"):
            lag = int(feature.split("_")[-1])
            records.append({"feature": feature, "max_source_offset": -lag, "known_future": False})
        elif feature.startswith("roll_"):
            records.append({"feature": feature, "max_source_offset": 0, "known_future": False})
        elif feature.startswith("target_"):
            # Target calendar and planned event features are future-dated but known in advance.
            records.append({"feature": feature, "max_source_offset": horizon, "known_future": True})
        elif feature in {"origin_time"}:
            records.append({"feature": feature, "max_source_offset": 0, "known_future": False})
        elif feature in {"target_time"}:
            records.append({"feature": feature, "max_source_offset": horizon, "known_future": True})
        elif feature.startswith("LEAK_"):
            records.append({"feature": feature, "max_source_offset": horizon + 3, "known_future": False})
        else:
            records.append({"feature": feature, "max_source_offset": None, "known_future": False})
    return pd.DataFrame(records)


def audit_feature_timing(provenance):
    issues = provenance[
        (provenance["max_source_offset"].fillna(999) > 0) &
        (~provenance["known_future"])
    ].copy()
    return issues

safe_provenance = build_feature_provenance(safe_features, HORIZON)
leaky_provenance = build_feature_provenance(leaky_features, HORIZON)

print("Safe feature timing issues:")
display(audit_feature_timing(safe_provenance))

print("Leaky feature timing issues:")
display(audit_feature_timing(leaky_provenance))


## 12. Residual diagnostics

An AI assistant may suggest a model, but residual diagnostics decide whether the model has left predictable structure. We compute the sample ACF and a simple Ljung-Box statistic.


In [ ]:
def sample_acf(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    denom = np.sum(x ** 2)
    values = []
    for lag in range(max_lag + 1):
        values.append(np.sum(x[lag:] * x[:len(x) - lag]) / denom)
    return np.asarray(values)


def ljung_box_statistic(residuals, lags=20):
    n = len(residuals)
    r = sample_acf(residuals, lags)
    q = n * (n + 2) * np.sum((r[1:] ** 2) / (n - np.arange(1, lags + 1)))
    p_value = 1 - chi2.cdf(q, df=lags)
    return q, p_value, r

# Use the best safe model among fitted candidates.
chosen_name = "gradient boosting safe"
chosen_pred = predictions[chosen_name]
residuals = y_test.to_numpy() - chosen_pred
q, p_value, residual_acf = ljung_box_statistic(residuals, lags=20)

print(f"Chosen model: {chosen_name}")
print(f"Residual mean: {residuals.mean():.3f}")
print(f"Residual std:  {residuals.std(ddof=1):.3f}")
print(f"Ljung-Box Q(20): {q:.2f}")
print(f"Approximate p-value: {p_value:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(test_safe["target_date"], residuals)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Test residuals over time")
ax.set_xlabel("target date")
ax.set_ylabel("observed - predicted")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
markerline, stemlines, baseline = ax.stem(np.arange(len(residual_acf)), residual_acf)
conf = 1.96 / np.sqrt(len(residuals))
ax.axhline(conf, linestyle="--")
ax.axhline(-conf, linestyle="--")
ax.set_title("Residual ACF")
ax.set_xlabel("lag")
ax.set_ylabel("ACF")
plt.show()


## 13. Workflow auditor as code

The next function formalizes an AI-style review. It does not replace statistical judgment, but it makes the checklist explicit and reproducible.


In [ ]:
@dataclass
class ForecastWorkflow:
    validation: str
    scaler_fit: str
    has_baseline: bool
    residual_diagnostics: bool
    future_covariates_known: bool
    reported_all_model_trials: bool
    used_leaky_features: bool


def audit_workflow(workflow):
    issues = []
    risk = 0

    if workflow.validation.lower() in {"random", "random split", "shuffle"}:
        issues.append("Use chronological or rolling-origin validation instead of a random split.")
        risk += 3

    if workflow.scaler_fit.lower() == "full data":
        issues.append("Fit scalers and imputers on each training block only.")
        risk += 2

    if not workflow.has_baseline:
        issues.append("Add naive and seasonal-naive baselines before claiming improvement.")
        risk += 2

    if not workflow.residual_diagnostics:
        issues.append("Check residual ACF, residual time plots, and horizon-wise errors.")
        risk += 2

    if not workflow.future_covariates_known:
        issues.append("Remove or replace covariates that are unavailable at the forecast origin.")
        risk += 3

    if not workflow.reported_all_model_trials:
        issues.append("Document tuning and model trials to reduce selection bias.")
        risk += 1

    if workflow.used_leaky_features:
        issues.append("Remove features that use target-time or post-target observations.")
        risk += 4

    if not issues:
        issues.append("No major issue detected. Still inspect plots, backtests, and limitations.")

    level = "low" if risk <= 2 else "medium" if risk <= 6 else "high"
    return {"risk_score": risk, "risk_level": level, "issues": issues}

workflows = {
    "safe workflow": ForecastWorkflow(
        validation="chronological",
        scaler_fit="train only",
        has_baseline=True,
        residual_diagnostics=True,
        future_covariates_known=True,
        reported_all_model_trials=True,
        used_leaky_features=False,
    ),
    "AI mistake example": ForecastWorkflow(
        validation="random split",
        scaler_fit="full data",
        has_baseline=False,
        residual_diagnostics=False,
        future_covariates_known=False,
        reported_all_model_trials=False,
        used_leaky_features=True,
    ),
}

for name, wf in workflows.items():
    result = audit_workflow(wf)
    print("\n" + name.upper())
    print("risk:", result["risk_score"], result["risk_level"])
    for issue in result["issues"]:
        print("-", issue)


## 14. Build a strong AI critic prompt from evidence

A weak prompt asks: "What is the best model?" A strong prompt gives the task, horizon, validation design, baselines, model results, and diagnostics. It asks for concrete verification steps.


In [ ]:
def build_critic_prompt(task, horizon, validation, information_set, metrics_table, residual_p_value):
    metrics_text = metrics_table.round(3).to_string(index=False)
    prompt = f"""
I am working on a time-series forecasting task.

Task: {task}
Forecast horizon: {horizon}
Validation design: {validation}
Information available at the forecast origin: {information_set}

Model and baseline results:
{metrics_text}

Residual diagnostic:
Approximate Ljung-Box p-value = {residual_p_value:.4f}

Please act as a skeptical reviewer. Identify possible leakage, invalid validation,
weak baselines, missing diagnostics, nonstationarity, and unsupported claims.
For each concern, give a concrete verification step that I can run using code,
a plot, a statistic, or a backtest. Do not recommend a more complex model unless
the evidence shows that the current residuals contain predictable structure.
""".strip()
    return prompt

critic_prompt = build_critic_prompt(
    task="seven-day-ahead daily demand forecasting",
    horizon=HORIZON,
    validation="chronological holdout with a future test block",
    information_set="target history through the origin date, trailing summaries, target-date calendar variables, and planned promotions/holidays known in advance",
    metrics_table=all_metrics,
    residual_p_value=p_value,
)

print(critic_prompt)


## 15. AI-assisted modeling log

A modeling log helps students distinguish assumptions, empirical evidence, diagnostics, and limitations. This is especially important when AI helps draft the report.


In [ ]:
log_rows = [
    {
        "category": "forecast task",
        "statement": "Forecast daily demand seven days ahead.",
        "evidence_or_check": "Target index equals origin plus seven in the unit test.",
        "decision": "Use chronological test block.",
    },
    {
        "category": "information set",
        "statement": "Features should be known at the forecast origin.",
        "evidence_or_check": "Feature provenance audit flags no safe-feature timing issues.",
        "decision": "Use lag, trailing rolling, calendar, promotion, and holiday features.",
    },
    {
        "category": "baseline",
        "statement": "Complex models must beat simple baselines.",
        "evidence_or_check": f"Best baseline RMSE = {baseline_metrics['RMSE'].min():.3f}.",
        "decision": "Report baseline comparison before model claims.",
    },
    {
        "category": "candidate model",
        "statement": f"Best observed workflow in the safe comparison: {best_model_name}.",
        "evidence_or_check": f"Best safe RMSE = {all_metrics['RMSE'].min():.3f} in the chronological test block.",
        "decision": "Treat this as empirical evidence, not proof of universal superiority.",
    },
    {
        "category": "diagnostics",
        "statement": "Residuals should be checked for remaining dependence.",
        "evidence_or_check": f"Approximate Ljung-Box p-value = {p_value:.4f}.",
        "decision": "If residual dependence remains, consider seasonal or dynamic extensions.",
    },
    {
        "category": "limitation",
        "statement": "The dataset is simulated and contains only two years of daily observations.",
        "evidence_or_check": "External validity is not tested.",
        "decision": "Do not overclaim production performance.",
    },
]

modeling_log = pd.DataFrame(log_rows)
modeling_log


## 16. Exercises

1. Change the forecast horizon from seven days to fourteen days. Which baseline becomes strongest?
2. Remove the target-date promotion and holiday features. Does performance change? How would you justify including or excluding these variables?
3. Add a rolling median feature using only past values. Confirm its timing with a unit test.
4. Create an intentionally invalid feature using a future target value. Confirm that the auditor flags it.
5. Replace the chronological holdout with rolling-origin backtesting. Compare the conclusions.
6. Write a critic prompt for a neural-network model. Include horizon-wise errors and residual diagnostics.
7. Draft a one-paragraph report that separates assumptions, evidence, limitations, and recommended next steps.

## 17. Mini-project

Choose a public or simulated time series. Build two workflows:

- a safe workflow with chronological or rolling-origin validation;
- an unsafe workflow with at least one known methodological error.

Then write a short audit report answering:

1. What is the forecasting origin and horizon?
2. What information is available at the origin?
3. Which features are valid and which are invalid?
4. What baselines did you compare against?
5. What residual diagnostics did you check?
6. Which AI-generated suggestions were useful?
7. Which AI-generated suggestions required correction?

## 18. AI-assisted study prompts

Use these prompts with an AI assistant after completing the lab.

**Prompt A: leakage review**

> Review my feature table for time leakage. The forecast horizon is seven days. At the forecast origin, I know target history, calendar variables, and planned promotions. For every feature, tell me whether it is valid, invalid, or needs clarification. Give a concrete test for each concern.

**Prompt B: validation review**

> I compared models using the metrics below. Explain whether my validation design answers a real forecasting question. Identify any missing baselines, residual checks, or horizon-specific diagnostics.

**Prompt C: report editor**

> Rewrite my model-comparison paragraph so that it separates mathematical assumptions, empirical evidence, diagnostics, limitations, and next steps. Do not claim causality or model superiority unless supported by validation evidence.
